# PySpark 8: real-world use cases and interviews

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/pyspark/pyspark-08-real-world-and-interviews.ipynb)

This last module ties the earlier ones together with five production patterns, a structure for PySpark interview answers, and a link to the hands-on practice.

**You will learn to:**

- Map a business need to a batch, streaming or backfill pattern
- Explain a PySpark design in a 30-second interview answer
- Know where the guided practice is

**Before you start:** PySpark 1 to 7.

## Setup and sample data

This module runs on the same four tables as PySpark 1, so each module can run on its own. The setup cell installs PySpark when it is missing and starts a local session.

In [ ]:
# Setup: installs this notebook's packages when they are missing (e.g. in Google Colab).
import importlib.util
import subprocess
import sys

PACKAGES = {"pyspark": "pyspark>=4.0,<4.1"}


def _missing(module: str, spec: str) -> bool:
    """True when the module is absent or its installed version does not satisfy spec."""
    try:
        if importlib.util.find_spec(module) is None:
            return True
    except ModuleNotFoundError:
        return True
    try:
        from importlib.metadata import PackageNotFoundError, version

        from packaging.requirements import Requirement
    except ImportError:
        return False
    requirement = Requirement(spec)
    try:
        return not requirement.specifier.contains(version(requirement.name), prereleases=True)
    except PackageNotFoundError:
        return False


missing = [spec for module, spec in PACKAGES.items() if _missing(module, spec)]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *missing])
print("ready:", ", ".join(PACKAGES))

In [ ]:
import os
import sys
import tempfile
import time

from pyspark.sql import SparkSession

# Python workers must use this notebook's interpreter, and every timestamp in the lessons is UTC.
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["TZ"] = "UTC"
time.tzset()
WORK_DIR = tempfile.mkdtemp(prefix="pyspark-lessons-")

spark = (
    SparkSession.builder
    .appName("pyspark-lessons")
    .master("local[2]")  # driver and executor share one JVM: the development topology
    .config("spark.sql.shuffle.partitions", "4")  # 200 is the default; tiny data needs few
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.warehouse.dir", os.path.join(WORK_DIR, "warehouse"))
    .config("spark.ui.enabled", "false")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version, "| shuffle partitions:", spark.conf.get("spark.sql.shuffle.partitions"))

## Sample data: the tables every lesson below runs on

This section seeds a small four-table slice of an orders platform, built with
`spark.createDataFrame` and an explicit schema per table. Three quirks the lessons lean on:
customer 4 has no `region`, customers 7 and 8 have never ordered, and one event is an exact
retry of the same row.

| table | grain | notable columns |
| --- | --- | --- |
| `customers` | one row per customer | `region`, `signup_date` (`DATE`) |
| `orders` | one row per order | `status`, `order_ts` (`TIMESTAMP`), `amount` (decimal) |
| `order_items` | one row per order line | `qty`, `unit_price` |
| `events` | one row per user event | `user_id`, `event`, `ts` |

In [ ]:
from datetime import date, datetime
from decimal import Decimal as D

from pyspark.sql import types as T

customers_schema = T.StructType([
    T.StructField("customer_id", T.IntegerType(), False),
    T.StructField("name", T.StringType(), False),
    T.StructField("region", T.StringType(), True),
    T.StructField("signup_date", T.DateType(), False),
])

customers = spark.createDataFrame(
    [
        (1, "Acme Corp",      "North", date(2024, 1, 3)),
        (2, "Bright Retail",  "North", date(2024, 2, 11)),
        (3, "Cyan Traders",   "South", date(2024, 3, 15)),
        (4, "Delta Grocers",  None,    date(2024, 5, 2)),
        (5, "Echo Supplies",  "West",  date(2024, 6, 20)),
        (6, "Foxtrot Foods",  "West",  date(2024, 8, 1)),
        (7, "Gamma Sports",   "East",  date(2024, 9, 14)),
        (8, "Hotel Homeware", "East",  date(2024, 11, 5)),
    ],
    schema=customers_schema,
)

customers.orderBy("customer_id").show()

In [ ]:
from pyspark.sql import types as T

orders_schema = T.StructType([
    T.StructField("order_id", T.IntegerType(), False),
    T.StructField("customer_id", T.IntegerType(), False),
    T.StructField("status", T.StringType(), True),
    T.StructField("order_ts", T.TimestampType(), False),
    T.StructField("amount", T.DecimalType(12, 2), True),
])

orders = spark.createDataFrame(
    [
        (1,  1, "paid",     datetime(2026, 1,  5,  9, 15), D("120.00")),
        (2,  1, "shipped",  datetime(2026, 1,  5, 10,  5),  D("84.50")),
        (3,  1, "returned", datetime(2026, 1,  5, 11, 40),  D("30.00")),
        (4,  2, "paid",     datetime(2026, 1,  6, 15, 20), D("260.00")),
        (5,  2, "placed",   datetime(2026, 1,  7,  8,  5),  D("55.25")),
        (6,  3, "shipped",  datetime(2026, 1,  7, 17, 45), D("149.00")),
        (7,  3, "paid",     datetime(2026, 1,  9, 12, 10), D("310.00")),
        (8,  4, "returned", datetime(2026, 1, 10, 19, 30),  D("89.90")),
        (9,  4, "shipped",  datetime(2026, 1, 12,  8, 55), D("1180.00")),
        (10, 5, "paid",     datetime(2026, 1, 12, 13,  0),   D("74.00")),
        (11, 6, "placed",   datetime(2026, 1, 13, 10, 25),  D("210.00")),
        (12, 6, "shipped",  datetime(2026, 1, 14,  0, 30),   D("96.40")),
    ],
    schema=orders_schema,
)

orders.orderBy("order_id").show()
orders.printSchema()

In [ ]:
from pyspark.sql import types as T

order_items_schema = T.StructType([
    T.StructField("order_id", T.IntegerType(), False),
    T.StructField("sku", T.StringType(), False),
    T.StructField("qty", T.IntegerType(), False),
    T.StructField("unit_price", T.DecimalType(8, 2), False),
])

order_items = spark.createDataFrame(
    [
        (1,  "SKU-4001",  2,  D("12.50")),
        (1,  "SKU-4007",  1,  D("95.00")),
        (2,  "SKU-4002",  1,  D("34.50")),
        (2,  "SKU-4009",  2,  D("25.00")),
        (3,  "SKU-4002",  1,  D("30.00")),
        (4,  "SKU-4003",  1, D("149.00")),
        (4,  "SKU-4004",  2,  D("55.50")),
        (5,  "SKU-4005",  1,  D("55.25")),
        (6,  "SKU-4003",  1, D("149.00")),
        (7,  "SKU-4006",  1, D("189.00")),
        (7,  "SKU-4008",  1, D("121.00")),
        (8,  "SKU-4010",  1,  D("89.90")),
        (9,  "SKU-4011", 20,  D("59.00")),
        (10, "SKU-4012",  2,  D("37.00")),
        (11, "SKU-4013",  1, D("210.00")),
        (12, "SKU-4014",  1,  D("41.40")),
        (12, "SKU-4005",  1,  D("55.00")),
    ],
    schema=order_items_schema,
)

order_items.orderBy("order_id", "sku").show(20)

In [ ]:
from pyspark.sql import types as T

events_schema = T.StructType([
    T.StructField("user_id", T.IntegerType(), False),
    T.StructField("event", T.StringType(), False),
    T.StructField("ts", T.TimestampType(), False),
])

events = spark.createDataFrame(
    [
        (1, "view",        datetime(2026, 1,  6,  9,  0)),
        (1, "add_to_cart", datetime(2026, 1,  6,  9,  2)),
        (1, "purchase",    datetime(2026, 1,  6,  9, 10)),
        (2, "view",        datetime(2026, 1,  7, 10,  0)),
        (2, "purchase",    datetime(2026, 1,  7, 10, 40)),
        (3, "view",        datetime(2026, 1,  6, 11, 15)),
        (3, "view",        datetime(2026, 1,  8, 11, 20)),
        (4, "add_to_cart", datetime(2026, 1,  9, 20,  5)),
        (4, "purchase",    datetime(2026, 1, 10,  8, 30)),
        (5, "view",        datetime(2026, 1, 11, 14, 45)),
        (5, "view",        datetime(2026, 1, 12,  9,  0)),
        (5, "purchase",    datetime(2026, 1, 12, 12, 30)),
        (5, "view",        datetime(2026, 1, 12,  9,  0)),
    ],
    schema=events_schema,
)

events.orderBy("user_id", "ts").show()

In [ ]:
from pyspark.sql import functions as F

frames = {
    "customers": customers,
    "orders": orders,
    "order_items": order_items,
    "events": events,
}
expected_rows = {"customers": 8, "orders": 12, "order_items": 17, "events": 13}

for table, frame in frames.items():
    actual = frame.count()
    assert actual == expected_rows[table], (table, actual, expected_rows[table])
    print(f"{table:<12} {actual:>2} rows")

per_order = (
    order_items
    .groupBy("order_id")
    .agg(F.sum(F.col("qty") * F.col("unit_price")).alias("lines"))
    .join(orders.select("order_id", "amount"), "order_id")
)
assert per_order.where(F.col("lines") != F.col("amount")).count() == 0
print("line totals reconcile with the orders ledger:", float(orders.agg(F.sum("amount")).first()[0]))

assert events.dropDuplicates().count() == events.count() - 1
print("and the retried event collapses to 12 distinct rows - seed verified")

In [ ]:
import json
from pyspark.sql import Window, functions as F
from pyspark.sql.functions import broadcast

In [ ]:
daily_revenue = (
    orders
    .where(F.col("status").isin("paid", "shipped"))
    .withColumn("order_date", F.to_date("order_ts"))
    .groupBy("order_date")
    .agg(F.sum("amount").alias("revenue"))
)
print(type(daily_revenue).__name__, "- nothing has executed yet")

In [ ]:
import contextlib
import io


def plan_of(df, mode="formatted"):
    """Return the text `df.explain(mode)` prints."""
    buffer = io.StringIO()
    with contextlib.redirect_stdout(buffer):
        df.explain(mode)
    return buffer.getvalue()


orders_path = os.path.join(WORK_DIR, "orders_parquet")
orders.write.mode("overwrite").parquet(orders_path)

result = (
    spark.read.parquet(orders_path)
    .where("order_ts >= timestamp '2026-01-10'")
    .select("order_id", "customer_id", "amount")
    .where("amount > 0")
)
plan = plan_of(result)
print(plan)
read_schema = next(line for line in plan.splitlines() if line.startswith("ReadSchema"))
assert "GreaterThan(amount,0.00)" in plan  # the filter reached the Parquet reader
assert "status" not in read_schema  # unused columns are never read

Five use cases show where the earlier modules' patterns pay off - nightly ELT, streaming ingestion,
feature tables, messy logs, safe backfills - then a section on how to say all of it in
an interview.

## Real world use cases for PySpark

### 1. Nightly batch ELT into a partitioned lake

Raw JSON lands in object storage, a Spark job cleans and types it, and the curated layer is
written as partitioned Parquet for downstream engines:

```python
orders = spark.read.schema(order_schema).json("s3://acme-raw/orders/ds=2026-10-04/")
(
    orders.where("amount_cents > 0")
    .withColumn("amount", F.col("amount_cents") / 100)
    .repartition("order_date")            # one file family per day
    .sortWithinPartitions("country")      # tight min/max for skipping
    .write.mode("overwrite")
    .option("compression", "zstd")
    .partitionBy("order_date")
    .parquet("s3://acme-curated/orders/")
)
```

Why Spark: a few minutes of distributed CPU replaces hours of single-node scripting, and explicit schemas remove inference surprises. Enable Adaptive Query Execution (`spark.sql.adaptive.enabled=true`) so skewed joins and partition counts self-tune.

### 2. Structured Streaming from Kafka into a lakehouse table

A streaming job decodes Kafka events, applies the batch transformations, and merges
micro-batches into an Iceberg or Delta table:

```python
(
    spark.readStream.format("kafka")
    .option("kafka.bootstrap.servers", "broker1:9092")
    .option("subscribe", "orders")
    .option("startingOffsets", "earliest")
    .load()
    .select(F.from_json(F.col("value").cast("string"), order_schema).alias("o"))
    .select("o.*")
    .writeStream
    .foreachBatch(merge_into_lakehouse)          # idempotent MERGE per batch
    .option("checkpointLocation", "s3://acme-checkpoints/orders/")
    .trigger(processingTime="1 minute")
    .start()
)
```

Why Spark: the batch and streaming APIs share transformations, so there is one business-logic implementation. Checkpointing plus an idempotent merge gives end-to-end exactly-once.

### 3. Feature engineering for machine learning

Feature tables use broadcast joins to small dimensions and vectorized pandas UDFs instead of
row-at-a-time Python UDFs:

```python
@pandas_udf("double")
def zscore(s: pd.Series) -> pd.Series:
    return (s - s.mean()) / s.std()

features = (
    events.join(F.broadcast(products), "product_id")
    .groupBy("customer_id")
    .agg(F.count("*").alias("orders_30d"), F.sum("amount").alias("spend_30d"))
    .withColumn("spend_z", zscore("spend_30d"))
)
```

Why Spark: the same feature definitions can be applied to historical training data and to daily scoring runs, which reduces train/serve skew.

### 4. Log and event processing with messy input

Semi-structured logs break schema assumptions. Read permissively (as in the corrupt-record
cell above), write good rows to the curated table and bad rows to a quarantine path, and evolve
the schema deliberately.

Why Spark: quarantine plus schema-on-read lets ingestion continue while the platform team fixes the producer, instead of failing the whole day.

### 5. Backfills and reprocessing without duplicating data

A bug fix requires reprocessing September. Set
`spark.sql.sources.partitionOverwriteMode=dynamic` and overwrite with only September's rows: the
dynamic-overwrite cell in PySpark 5 shows exactly this on one day.

Why Spark: the same job that handles daily loads can reprocess a month, and dynamic overwrite makes the rerun idempotent at partition granularity - the property that makes backfills safe.

## What should you say in a PySpark interview answer?

A strong PySpark interview answer should include the API, the execution model, and the production concern.

Example answer for "How do you deduplicate orders?": `row_number()` over
`Window.partitionBy("order_id").orderBy(F.col("source_lsn").desc())`, keep `rn = 1` (the CDC
cell above), then explain:

+ `row_number` keeps one row per business key.
+ Ordering must use source metadata, not random ingestion order.
+ The result should be merged into the target for idempotency.
+ If one key is extremely hot, this window can skew and needs tuning.

How it works under the hood, restated as an answer structure. A strong answer names the API, the execution model, and the failure mode. For deduplication: "I use `row_number()` over `Window.partitionBy(business_key).orderBy(source_lsn desc)` filtered to `rn = 1`, then `MERGE`; that forces one shuffle by key, and if one key is hot the window skews, so I check the partition sizes and salt or let AQE skew join split it."

Worked example, a 30-second answer skeleton:

```text
1. API       - the exact functions and window spec you call
2. Engine    - narrow vs wide, where the shuffle is, partition count
3. Ops       - idempotency, skew, cost, monitoring, failure recovery
```

Common mistake: answering with syntax only. Interviewers probe the plan, so tie every method to a shuffle, a partition count, or a failure mode.

## Practice: PySpark concepts

The hands-on practice for this topic lives in its own notebook: [`04_pyspark_concepts.ipynb`](pyspark-concepts-end-to-end.ipynb) — [Open in Colab](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/pyspark/pyspark-concepts-end-to-end.ipynb).

## Check yourself

The lessons above end with takeaways. Re-run each cell in order, and use `explain` on any query you do not understand before you move on.

## Key takeaways

- Name the API, the execution model and the cost before you name the code.
- Backfills reuse the daily job with a date parameter and dynamic overwrite.
- Practise the answer structure on the interview notebooks in this handbook.